# Real Estate Price Prediction — Model Training Notebook
**GROUP 15 | CSC 476 | University of Ibadan**

This notebook covers:
1. Loading and exploring the housing dataset
2. Preprocessing (encoding binary and categorical features)
3. Training Linear Regression, Decision Tree, and Random Forest
4. Evaluating each model with MAE, RMSE, and R²
5. Saving all models and best_model.pkl

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All imports successful')

## 2. Load Dataset

In [ ]:
# Adjust path if running from a different directory
DATASET_PATH = '../dataset/housing.csv'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nBasic statistics:')
df.describe()

In [ ]:
# Distribution of target variable
plt.figure(figsize=(8, 4))
plt.hist(df['price'], bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Property Prices')
plt.xlabel('Price')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 6))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# Encode yes/no binary columns as 1/0
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# Encode furnishingstatus as ordinal
furnish_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df['furnishingstatus'] = df['furnishingstatus'].map(furnish_map)

print('Encoding complete. Sample:')
df.head()

In [ ]:
FEATURES = ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom',
            'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']
TARGET = 'price'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5. Train and Evaluate Models

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f'\n{name}')
    print(f'  MAE:  {mae:>15,.2f}')
    print(f'  RMSE: {rmse:>15,.2f}')
    print(f'  R²:   {r2:>15.4f}')
    return {'name': name, 'model': model, 'mae': mae, 'rmse': rmse, 'r2': r2}

results = []
results.append(evaluate_model('Linear Regression',   LinearRegression(),                         X_train, X_test, y_train, y_test))
results.append(evaluate_model('Decision Tree',        DecisionTreeRegressor(random_state=42),     X_train, X_test, y_train, y_test))
results.append(evaluate_model('Random Forest',        RandomForestRegressor(n_estimators=100, random_state=42), X_train, X_test, y_train, y_test))

## 6. Model Comparison

In [ ]:
comparison = pd.DataFrame([{'Model': r['name'], 'MAE': r['mae'], 'RMSE': r['rmse'], 'R²': r['r2']} for r in results])
comparison = comparison.sort_values('R²', ascending=False).reset_index(drop=True)
print('\nModel Comparison (sorted by R²):')
display(comparison)

# Bar chart of R² scores
plt.figure(figsize=(7, 4))
plt.bar(comparison['Model'], comparison['R²'], color=['#2196F3', '#FF9800', '#4CAF50'])
plt.title('R² Score by Model')
plt.ylabel('R²')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
filename_map = {
    'Linear Regression': 'linear_regression.pkl',
    'Decision Tree':     'decision_tree.pkl',
    'Random Forest':     'random_forest.pkl',
}

for r in results:
    path = os.path.join(MODELS_DIR, filename_map[r['name']])
    joblib.dump(r['model'], path)
    print(f'Saved: {path}')

# Save best model separately
best = max(results, key=lambda r: r['r2'])
best_path = os.path.join(MODELS_DIR, 'best_model.pkl')
joblib.dump(best['model'], best_path)
print(f"\nBest model: {best['name']} (R²={best['r2']:.4f})")
print(f'Saved as: {best_path}')

## 8. Verify predict_price() Function

In [ ]:
import sys
sys.path.insert(0, '..')
from predict import predict_price

test_input = {
    'area': 3000,
    'bedrooms': 3,
    'bathrooms': 2,
    'stories': 1,
    'mainroad': 'yes',
    'guestroom': 'no',
    'basement': 'no',
    'hotwaterheating': 'no',
    'airconditioning': 'yes',
    'parking': 1,
    'prefarea': 'no',
    'furnishingstatus': 'furnished',
}

price = predict_price(test_input)
print(f'Test prediction: {price:,.2f}')
print('\n✅ predict_price() is working correctly.')